# Forecast Model Robustness and Selection

## Phase 6.4 — Business Objective

Evaluate whether the leading retail sales forecasting models
perform consistently across forecast horizons, historical
validation windows, and calendar periods.

The objective is to support an evidence-based model selection
decision rather than choosing a model using overall WAPE alone.

## Existing Forecasting Results

| Model | WAPE | RMSE | Bias |
|---|---:|---:|---:|
| 50/50 Blended | 8.36% | 2,737,131.72 | -6.45% |
| ETS Additive | 8.48% | 2,542,024.93 | +0.96% |
| Calendar Only Ridge | 10.01% | 3,007,249.48 | -7.85% |
| Calendar + Promotion Ridge | 12.24% | 3,789,357.27 | -1.34% |

## Robustness Questions

1. Which model performs best at each forecast horizon?
2. Which model performs consistently across rolling-origin windows?
3. Where are the largest forecast errors concentrated?
4. How frequently does each model outperform the alternatives?
5. Does the model ranking change across historical periods?
6. Which model is most appropriate for retail business planning?

## Validation Design

- Historical period: January 2013–July 2017
- Complete monthly observations: 55
- Minimum training history: 36 months
- Rolling-origin windows: 17
- Forecast horizon: 1–3 months
- Matched forecast cases: 51 per model
- Evaluation metrics: MAE, RMSE, WAPE, and Bias

## Data Governance Rules

1. Use previously validated forecast predictions.
2. Compare models using identical forecast cases.
3. Preserve chronological forecast origins.
4. Do not retrain models during the initial robustness investigation.
5. Do not use future actual sales as forecasting predictors.
6. Distinguish pooled metrics from window-level averages.
7. Treat overlapping forecast cases as dependent observations.
8. Do not interpret historical backtest performance as guaranteed future accuracy.

## Expected Deliverables

- Forecast horizon performance comparison
- Rolling-window robustness analysis
- Model win-rate analysis
- Forecast error concentration analysis
- Business-oriented model selection recommendation
- Validated CSV reports
- Updated project README and GitHub milestone

## Step 1 — Load Existing Forecast Predictions

Load previously validated benchmark and Ridge model forecasts.

The purpose is to reuse historical predictions for robustness
analysis without retraining the models.

Before comparing model performance, verify that both datasets
contain the expected records, windows, and forecast horizons.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the project root automatically
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "processed").exists()
)

# Define existing report paths
report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_evaluation"
)

benchmark_path = (
    report_dir / "seven_model_matched_forecasts.csv"
)

ridge_path = (
    report_dir / "ridge_rolling_origin_forecasts.csv"
)

# Load the saved reports
benchmark_df = pd.read_csv(benchmark_path)

ridge_df = pd.read_csv(ridge_path)

# Validate expected record counts
assert len(benchmark_df) == 51
assert len(ridge_df) == 102

# Validate forecast windows
assert benchmark_df["window_id"].nunique() == 17
assert ridge_df["window_id"].nunique() == 17

# Validate Ridge model coverage
assert ridge_df["model"].nunique() == 2

# Validate Ridge forecast horizons
assert set(ridge_df["forecast_horizon"]) == {1, 2, 3}

print("Forecast report loading: PASSED")

print("Benchmark forecast cases:", len(benchmark_df))
print("Ridge forecast records:", len(ridge_df))

print("Benchmark windows:", benchmark_df["window_id"].nunique())
print("Ridge windows:", ridge_df["window_id"].nunique())

print("\nBenchmark columns:")
print(benchmark_df.columns.tolist())

print("\nRidge columns:")
print(ridge_df.columns.tolist())

Forecast report loading: PASSED
Benchmark forecast cases: 51
Ridge forecast records: 102
Benchmark windows: 17
Ridge windows: 17

Benchmark columns:
['window_id', 'train_end', 'month', 'total_sales', 'naive_forecast', 'seasonal_naive_forecast', 'blended_forecast', 'ets_forecast', 'damped_ets_forecast', 'seasonal_index_forecast', 'trend_adjusted_forecast']

Ridge columns:
['window_id', 'train_end', 'forecast_month', 'forecast_horizon', 'model', 'actual_sales', 'predicted_sales', 'training_examples', 'forecast_error', 'absolute_error', 'squared_error']


## Step 2 — Standardize Forecast Dates and Validate Matched Cases

Convert forecast dates into consistent datetime types.

Derive the benchmark forecast horizon from its forecast
origin and target month.

Verify that benchmark and Ridge forecasts refer to identical
rolling-origin windows, forecast months, horizons, and actual
sales values.

Do not combine model predictions until these checks pass.

In [2]:
# Work on copies to preserve the original loaded reports
benchmark_work_df = benchmark_df.copy()
ridge_work_df = ridge_df.copy()

# Standardize date columns
benchmark_work_df["train_end"] = pd.to_datetime(
    benchmark_work_df["train_end"]
)

benchmark_work_df["month"] = pd.to_datetime(
    benchmark_work_df["month"]
)

ridge_work_df["train_end"] = pd.to_datetime(
    ridge_work_df["train_end"]
)

ridge_work_df["forecast_month"] = pd.to_datetime(
    ridge_work_df["forecast_month"]
)

# Calculate months between the origin and target
benchmark_work_df["forecast_horizon"] = (
    (
        benchmark_work_df["month"].dt.year
        - benchmark_work_df["train_end"].dt.year
    ) * 12
    + (
        benchmark_work_df["month"].dt.month
        - benchmark_work_df["train_end"].dt.month
    )
)

# Standardize column names
benchmark_work_df = benchmark_work_df.rename(
    columns={
        "month": "forecast_month",
        "total_sales": "actual_sales",
    }
)

# Define the unique forecast case
match_columns = [
    "window_id",
    "train_end",
    "forecast_month",
    "forecast_horizon",
]

# Validate uniqueness
assert not benchmark_work_df.duplicated(
    subset=match_columns
).any()

assert not ridge_work_df.duplicated(
    subset=match_columns + ["model"]
).any()

# Confirm benchmark horizons
assert set(
    benchmark_work_df["forecast_horizon"]
) == {1, 2, 3}

# Each benchmark case must have two Ridge predictions
ridge_case_counts = ridge_work_df.groupby(
    match_columns
).size()

assert len(ridge_case_counts) == 51
assert ridge_case_counts.eq(2).all()

# Match benchmark cases with Ridge cases
matched_cases_df = benchmark_work_df[
    match_columns + ["actual_sales"]
].merge(
    ridge_work_df[
        match_columns + ["model", "actual_sales"]
    ],
    on=match_columns,
    how="outer",
    suffixes=("_benchmark", "_ridge"),
    indicator=True,
    validate="one_to_many",
)

# Validate all cases matched
assert matched_cases_df["_merge"].eq("both").all()

# Validate actual sales reconciliation
sales_difference = (
    matched_cases_df["actual_sales_benchmark"]
    - matched_cases_df["actual_sales_ridge"]
).abs()

assert np.allclose(
    matched_cases_df["actual_sales_benchmark"],
    matched_cases_df["actual_sales_ridge"],
    atol=0.01,
    rtol=0,
)

print("Forecast case matching: PASSED")
print("Unique benchmark cases:", len(benchmark_work_df))
print("Unique Ridge cases:", len(ridge_case_counts))
print("Matched model-case records:", len(matched_cases_df))
print("Unmatched records:", matched_cases_df["_merge"].ne("both").sum())
print("Maximum actual sales difference:", sales_difference.max())

print("\nBenchmark forecast horizon distribution:")
print(
    benchmark_work_df["forecast_horizon"]
    .value_counts()
    .sort_index()
)

Forecast case matching: PASSED
Unique benchmark cases: 51
Unique Ridge cases: 51
Matched model-case records: 102
Unmatched records: 0
Maximum actual sales difference: 3.725290298461914e-09

Benchmark forecast horizon distribution:
forecast_horizon
1    17
2    17
3    17
Name: count, dtype: int64


## Step 3 — Build the Unified Nine-Model Forecast Dataset

Convert the seven benchmark predictions from wide format
to long format and combine them with the two Ridge models.

Expected structure:
- 9 forecasting models
- 51 matched cases per model
- 459 total model-case records
- 17 rolling-origin windows
- 3 forecast horizons

Validate uniqueness, actual sales consistency, and
forecast value integrity before analyzing robustness.

In [4]:
# Map benchmark forecast columns to readable model names
benchmark_model_columns = {
    "naive_forecast": "Naive",
    "seasonal_naive_forecast": "Seasonal Naive",
    "blended_forecast": "50/50 Blended",
    "ets_forecast": "ETS Additive",
    "damped_ets_forecast": "ETS Damped Trend",
    "seasonal_index_forecast": "Seasonal Index",
    "trend_adjusted_forecast": "Trend-Adjusted Seasonal Index",
}

# Convert benchmark forecasts from wide to long format
benchmark_long_df = benchmark_work_df.melt(
    id_vars=match_columns + ["actual_sales"],
    value_vars=list(benchmark_model_columns.keys()),
    var_name="model",
    value_name="predicted_sales",
)

# Replace column names with readable model names
benchmark_long_df["model"] = (
    benchmark_long_df["model"]
    .map(benchmark_model_columns)
)

# Standardize Ridge forecast columns
ridge_long_df = ridge_work_df[
    match_columns + [
        "actual_sales",
        "model",
        "predicted_sales",
    ]
].copy()

# Combine all nine forecasting models
unified_forecasts_df = pd.concat(
    [benchmark_long_df, ridge_long_df],
    ignore_index=True,
)

# Sort for easier investigation
unified_forecasts_df = unified_forecasts_df.sort_values(
    match_columns + ["model"]
).reset_index(drop=True)

# Validate the complete dataset
assert len(unified_forecasts_df) == 459

assert unified_forecasts_df["model"].nunique() == 9

assert unified_forecasts_df["window_id"].nunique() == 17

assert not unified_forecasts_df.duplicated(
    subset=match_columns + ["model"]
).any()

# Every model must contain exactly 51 forecast cases
model_counts = unified_forecasts_df.groupby(
    "model"
).size()

assert model_counts.eq(51).all()

# Each matched forecast case must contain nine models
case_counts = unified_forecasts_df.groupby(
    match_columns
)["model"].nunique()

assert len(case_counts) == 51
assert case_counts.eq(9).all()

# Actual sales must be identical within each forecast case
actual_sales_range = unified_forecasts_df.groupby(
    match_columns
)["actual_sales"].agg(
    lambda values: values.max() - values.min()
)

assert (actual_sales_range < 0.01).all()

# Validate sales and forecast values
assert unified_forecasts_df[
    ["actual_sales", "predicted_sales"]
].notna().all().all()

assert np.isfinite(
    unified_forecasts_df[
        ["actual_sales", "predicted_sales"]
    ].to_numpy()
).all()

assert (
    unified_forecasts_df["predicted_sales"] >= 0
).all()

print("Unified nine-model dataset validation: PASSED")

print("Total model-case records:", len(unified_forecasts_df))
print("Unique models:", unified_forecasts_df["model"].nunique())
print("Unique forecast cases:", len(case_counts))
print("Rolling-origin windows:", unified_forecasts_df["window_id"].nunique())
print("Maximum actual sales difference:", actual_sales_range.max())

print("\nRecords per model:")
print(model_counts.to_string())

print("\nUnified dataset preview:")
display(unified_forecasts_df.head(10))

Unified nine-model dataset validation: PASSED
Total model-case records: 459
Unique models: 9
Unique forecast cases: 51
Rolling-origin windows: 17
Maximum actual sales difference: 3.725290298461914e-09

Records per model:
model
50/50 Blended                    51
Calendar + Promotion             51
Calendar Only                    51
ETS Additive                     51
ETS Damped Trend                 51
Naive                            51
Seasonal Index                   51
Seasonal Naive                   51
Trend-Adjusted Seasonal Index    51

Unified dataset preview:


,window_id,train_end,forecast_month,forecast_horizon,actual_sales,model,predicted_sales
0,1,2015-12-01,2016-01-01,1,2.397780e+07,50/50 Blended,2.107045e+07
1,1,2015-12-01,2016-01-01,1,2.397780e+07,Calendar + Promotion,1.615498e+07
2,1,2015-12-01,2016-01-01,1,2.397780e+07,Calendar Only,1.747551e+07
3,1,2015-12-01,2016-01-01,1,2.397780e+07,ETS Additive,2.346665e+07
4,1,2015-12-01,2016-01-01,1,2.397780e+07,ETS Damped Trend,2.262673e+07
5,1,2015-12-01,2016-01-01,1,2.397780e+07,Naive,2.724398e+07
6,1,2015-12-01,2016-01-01,1,2.397780e+07,Seasonal Index,1.812011e+07
7,1,2015-12-01,2016-01-01,1,2.397780e+07,Seasonal Naive,1.489692e+07
8,1,2015-12-01,2016-01-01,1,2.397780e+07,Trend-Adjusted Seasonal Index,2.083680e+07
9,1,2015-12-01,2016-02-01,2,2.194741e+07,50/50 Blended,2.049319e+07


## Step 4 — Calculate and Reconcile Nine-Model Forecast Accuracy

Calculate MAE, RMSE, WAPE, and Bias independently from
the unified 459-record forecasting dataset.

Compare the calculated results with previously validated
benchmark and Ridge accuracy reports.

All nine models must be evaluated on the same 51 matched
forecast cases.

The results represent historical rolling-origin backtest
performance, not independent final-test performance.

In [5]:
# Preserve the validated unified dataset
accuracy_work_df = unified_forecasts_df.copy()

# Calculate individual forecast errors
accuracy_work_df["forecast_error"] = (
    accuracy_work_df["predicted_sales"]
    - accuracy_work_df["actual_sales"]
)

accuracy_work_df["absolute_error"] = (
    accuracy_work_df["forecast_error"].abs()
)

accuracy_work_df["squared_error"] = (
    accuracy_work_df["forecast_error"] ** 2
)

# Calculate pooled accuracy metrics for each model
nine_model_accuracy_df = (
    accuracy_work_df
    .groupby("model")
    .agg(
        forecast_cases=("forecast_error", "size"),
        MAE=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        total_absolute_error=("absolute_error", "sum"),
        total_forecast_error=("forecast_error", "sum"),
        total_actual_sales=("actual_sales", "sum"),
    )
    .reset_index()
)

# Calculate RMSE
nine_model_accuracy_df["RMSE"] = np.sqrt(
    nine_model_accuracy_df["mean_squared_error"]
)

# Calculate pooled WAPE
nine_model_accuracy_df["WAPE_pct"] = (
    nine_model_accuracy_df["total_absolute_error"]
    / nine_model_accuracy_df["total_actual_sales"]
    * 100
)

# Calculate pooled forecast bias
nine_model_accuracy_df["Bias_pct"] = (
    nine_model_accuracy_df["total_forecast_error"]
    / nine_model_accuracy_df["total_actual_sales"]
    * 100
)

# Keep the final reporting columns
nine_model_accuracy_df = nine_model_accuracy_df[
    [
        "model",
        "forecast_cases",
        "MAE",
        "RMSE",
        "WAPE_pct",
        "Bias_pct",
    ]
].sort_values(
    "WAPE_pct"
).reset_index(drop=True)

# Validate all nine models
assert len(nine_model_accuracy_df) == 9

assert nine_model_accuracy_df[
    "forecast_cases"
].eq(51).all()

assert np.isfinite(
    nine_model_accuracy_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

# Display results
print("Nine-model accuracy calculation: PASSED")

print("\nNine-model accuracy comparison:")

display(
    nine_model_accuracy_df.style.format(
        {
            "MAE": "{:,.2f}",
            "RMSE": "{:,.2f}",
            "WAPE_pct": "{:.2f}%",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

Nine-model accuracy calculation: PASSED

Nine-model accuracy comparison:


,model,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,50/50 Blended,51,"2,067,404.61","2,737,131.72",8.36%,-6.45%
1,ETS Additive,51,"2,098,242.04","2,542,024.93",8.48%,+0.96%
2,ETS Damped Trend,51,"2,118,859.66","2,549,228.14",8.56%,+0.30%
3,Naive,51,"2,178,014.34","2,735,640.76",8.80%,-0.59%
4,Calendar Only,51,"2,475,681.41","3,007,249.48",10.01%,-7.85%
5,Calendar + Promotion,51,"3,028,155.12","3,789,357.27",12.24%,-1.34%
6,Seasonal Naive,51,"3,210,528.38","4,529,398.49",12.98%,-12.31%
7,Seasonal Index,51,"3,591,288.90","4,136,902.96",14.52%,-8.09%
8,Trend-Adjusted Seasonal Index,51,"4,786,033.22","6,151,133.27",19.35%,+12.58%


## Step 5 — Forecast Horizon Robustness Analysis

Evaluate the nine forecasting models separately for
one-month, two-month, and three-month forecast horizons.

Each model must contain 17 forecast cases per horizon.

Calculate MAE, RMSE, pooled WAPE, and pooled Bias using
the same error definitions as the overall model comparison.

Investigate whether model rankings change by horizon.

In [6]:
# Group forecast errors by model and forecast horizon
horizon_accuracy_df = (
    accuracy_work_df
    .groupby(["model", "forecast_horizon"])
    .agg(
        forecast_cases=("forecast_error", "size"),
        MAE=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        total_absolute_error=("absolute_error", "sum"),
        total_forecast_error=("forecast_error", "sum"),
        total_actual_sales=("actual_sales", "sum"),
    )
    .reset_index()
)

# Calculate RMSE
horizon_accuracy_df["RMSE"] = np.sqrt(
    horizon_accuracy_df["mean_squared_error"]
)

# Calculate pooled WAPE
horizon_accuracy_df["WAPE_pct"] = (
    horizon_accuracy_df["total_absolute_error"]
    / horizon_accuracy_df["total_actual_sales"]
    * 100
)

# Calculate pooled Bias
horizon_accuracy_df["Bias_pct"] = (
    horizon_accuracy_df["total_forecast_error"]
    / horizon_accuracy_df["total_actual_sales"]
    * 100
)

# Keep reporting columns
horizon_accuracy_df = horizon_accuracy_df[
    [
        "model",
        "forecast_horizon",
        "forecast_cases",
        "MAE",
        "RMSE",
        "WAPE_pct",
        "Bias_pct",
    ]
].sort_values(
    ["forecast_horizon", "WAPE_pct"]
).reset_index(drop=True)

# Validate expected structure
assert len(horizon_accuracy_df) == 27

assert horizon_accuracy_df[
    "forecast_cases"
].eq(17).all()

assert horizon_accuracy_df[
    "model"
].nunique() == 9

assert set(
    horizon_accuracy_df["forecast_horizon"]
) == {1, 2, 3}

assert np.isfinite(
    horizon_accuracy_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

# Display horizon WAPE comparison
horizon_wape_df = horizon_accuracy_df.pivot(
    index="model",
    columns="forecast_horizon",
    values="WAPE_pct",
)

horizon_wape_df.columns = [
    "Horizon 1",
    "Horizon 2",
    "Horizon 3",
]

# Order models by overall WAPE ranking
model_order = nine_model_accuracy_df["model"].tolist()

horizon_wape_df = horizon_wape_df.loc[model_order]

print("Forecast horizon robustness validation: PASSED")
print("Model-horizon combinations:", len(horizon_accuracy_df))
print("Forecast cases per combination: 17")

print("\nWAPE by forecast horizon (%):")

display(
    horizon_wape_df.style.format("{:.2f}%")
)

print("\nBest model by WAPE for each horizon:")

best_by_horizon_df = (
    horizon_accuracy_df
    .sort_values(["forecast_horizon", "WAPE_pct"])
    .groupby("forecast_horizon")
    .head(1)
    [["forecast_horizon", "model", "WAPE_pct", "Bias_pct"]]
)

display(
    best_by_horizon_df.style.format(
        {
            "WAPE_pct": "{:.2f}%",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

Forecast horizon robustness validation: PASSED
Model-horizon combinations: 27
Forecast cases per combination: 17

WAPE by forecast horizon (%):


,Horizon 1,Horizon 2,Horizon 3
model,,,
50/50 Blended,8.70%,8.79%,7.59%
ETS Additive,7.96%,8.71%,8.77%
ETS Damped Trend,7.76%,8.70%,9.22%
Naive,8.20%,9.32%,8.89%
Calendar Only,10.45%,9.96%,9.61%
Calendar + Promotion,11.70%,12.18%,12.83%
Seasonal Naive,14.33%,12.94%,11.69%
Seasonal Index,14.70%,14.74%,14.12%
Trend-Adjusted Seasonal Index,19.69%,19.58%,18.77%



Best model by WAPE for each horizon:


,forecast_horizon,model,WAPE_pct,Bias_pct
0,1,ETS Damped Trend,7.76%,-0.30%
9,2,ETS Damped Trend,8.70%,+0.14%
18,3,50/50 Blended,7.59%,-6.27%


## Step 6 — Rolling-Window Forecast Robustness

Evaluate forecast accuracy separately for each of the
17 historical rolling-origin windows.

Each model must have three forecast cases per window.

Calculate window-level MAE, RMSE, WAPE, and Bias.

Investigate variation in historical model performance
and identify periods with unusually high forecast errors.

Window-level metrics are descriptive because forecast
targets overlap across adjacent rolling-origin windows.

In [7]:
# Calculate accuracy separately for each model and window
window_accuracy_df = (
    accuracy_work_df
    .groupby(["window_id", "train_end", "model"])
    .agg(
        forecast_cases=("forecast_error", "size"),
        MAE=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        total_absolute_error=("absolute_error", "sum"),
        total_forecast_error=("forecast_error", "sum"),
        total_actual_sales=("actual_sales", "sum"),
    )
    .reset_index()
)

# Calculate window-level RMSE
window_accuracy_df["RMSE"] = np.sqrt(
    window_accuracy_df["mean_squared_error"]
)

# Calculate window-level WAPE
window_accuracy_df["WAPE_pct"] = (
    window_accuracy_df["total_absolute_error"]
    / window_accuracy_df["total_actual_sales"]
    * 100
)

# Calculate window-level Bias
window_accuracy_df["Bias_pct"] = (
    window_accuracy_df["total_forecast_error"]
    / window_accuracy_df["total_actual_sales"]
    * 100
)

# Keep reporting columns
window_accuracy_df = window_accuracy_df[
    [
        "window_id",
        "train_end",
        "model",
        "forecast_cases",
        "MAE",
        "RMSE",
        "WAPE_pct",
        "Bias_pct",
    ]
].sort_values(
    ["window_id", "WAPE_pct"]
).reset_index(drop=True)

# Validate expected records
assert len(window_accuracy_df) == 153

assert window_accuracy_df["model"].nunique() == 9

assert window_accuracy_df["window_id"].nunique() == 17

assert window_accuracy_df["forecast_cases"].eq(3).all()

assert not window_accuracy_df.duplicated(
    subset=["window_id", "model"]
).any()

assert np.isfinite(
    window_accuracy_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

# Summarize model robustness across windows
window_robustness_df = (
    window_accuracy_df
    .groupby("model")
    .agg(
        windows=("window_id", "count"),
        mean_window_WAPE=("WAPE_pct", "mean"),
        median_window_WAPE=("WAPE_pct", "median"),
        std_window_WAPE=("WAPE_pct", "std"),
        min_window_WAPE=("WAPE_pct", "min"),
        max_window_WAPE=("WAPE_pct", "max"),
    )
    .reset_index()
)

# Preserve the overall WAPE ranking
window_robustness_df = (
    window_robustness_df
    .set_index("model")
    .loc[model_order]
    .reset_index()
)

print("Rolling-window robustness validation: PASSED")
print("Model-window combinations:", len(window_accuracy_df))
print("Windows per model:", window_robustness_df["windows"].unique())
print("Forecast cases per model-window: 3")

print("\nRolling-window robustness summary (%):")

display(
    window_robustness_df.style.format(
        {
            "mean_window_WAPE": "{:.2f}%",
            "median_window_WAPE": "{:.2f}%",
            "std_window_WAPE": "{:.2f}",
            "min_window_WAPE": "{:.2f}%",
            "max_window_WAPE": "{:.2f}%",
        }
    )
)

Rolling-window robustness validation: PASSED
Model-window combinations: 153
Windows per model: [17]
Forecast cases per model-window: 3

Rolling-window robustness summary (%):


,model,windows,mean_window_WAPE,median_window_WAPE,std_window_WAPE,min_window_WAPE,max_window_WAPE
0,50/50 Blended,17,8.33%,7.27%,5.72,2.48%,22.76%
1,ETS Additive,17,8.69%,7.03%,4.66,2.88%,14.41%
2,ETS Damped Trend,17,8.78%,7.38%,4.64,3.11%,15.56%
3,Naive,17,8.78%,9.12%,4.83,2.49%,18.35%
4,Calendar Only,17,10.01%,9.52%,5.99,1.21%,24.89%
5,Calendar + Promotion,17,12.65%,9.70%,9.58,3.09%,33.85%
6,Seasonal Naive,17,13.13%,7.64%,12.16,2.16%,37.65%
7,Seasonal Index,17,14.51%,14.13%,6.10,3.70%,26.29%
8,Trend-Adjusted Seasonal Index,17,19.87%,14.66%,15.06,3.23%,43.84%


## Step 7 — Investigate Worst Rolling-Origin Windows

Identify the three highest-WAPE historical forecast windows
for each model.

Use the existing validated window-level results.

Inspect the forecast origin, WAPE, RMSE, and Bias to
understand when forecast accuracy deteriorated.

Do not assume that high forecast errors were caused by
promotions, seasonality, or operational changes without
additional supporting evidence.

In [8]:
# Identify the three highest-WAPE windows per model
worst_windows_df = (
    window_accuracy_df
    .sort_values(
        ["model", "WAPE_pct"],
        ascending=[True, False]
    )
    .groupby("model")
    .head(3)
    .copy()
)

# Rank worst windows within each model
worst_windows_df["worst_rank"] = (
    worst_windows_df
    .groupby("model")["WAPE_pct"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Select useful diagnostic columns
worst_windows_df = worst_windows_df[
    [
        "model",
        "worst_rank",
        "window_id",
        "train_end",
        "WAPE_pct",
        "RMSE",
        "Bias_pct",
    ]
].sort_values(
    ["model", "worst_rank"]
).reset_index(drop=True)

# Validation
assert len(worst_windows_df) == 27

assert worst_windows_df["model"].nunique() == 9

assert worst_windows_df.groupby(
    "model"
).size().eq(3).all()

assert set(worst_windows_df["worst_rank"]) == {1, 2, 3}

print("Worst-window diagnostics validation: PASSED")
print("Total diagnostic records:", len(worst_windows_df))

# Focus first on the leading models
leading_models = [
    "50/50 Blended",
    "ETS Additive",
    "ETS Damped Trend",
]

print("\nWorst three windows for the leading models:")

display(
    worst_windows_df[
        worst_windows_df["model"].isin(leading_models)
    ].style.format(
        {
            "train_end": "{:%Y-%m}",
            "WAPE_pct": "{:.2f}%",
            "RMSE": "{:,.2f}",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

print("\nWorst window for each of the nine models:")

display(
    worst_windows_df[
        worst_windows_df["worst_rank"] == 1
    ].style.format(
        {
            "train_end": "{:%Y-%m}",
            "WAPE_pct": "{:.2f}%",
            "RMSE": "{:,.2f}",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

Worst-window diagnostics validation: PASSED
Total diagnostic records: 27

Worst three windows for the leading models:


,model,worst_rank,window_id,train_end,WAPE_pct,RMSE,Bias_pct
0,50/50 Blended,1,3,2016-02,22.76%,"5,768,732.10",-22.76%
1,50/50 Blended,2,2,2016-01,18.20%,"4,579,542.49",-18.20%
2,50/50 Blended,3,4,2016-03,15.67%,"4,719,325.46",-15.22%
9,ETS Additive,1,9,2016-08,14.41%,"3,474,959.84",+14.41%
10,ETS Additive,2,7,2016-06,14.19%,"3,502,985.27",+14.19%
11,ETS Additive,3,6,2016-05,13.86%,"3,196,083.80",+13.86%
12,ETS Damped Trend,1,6,2016-05,15.56%,"3,566,475.04",+15.56%
13,ETS Damped Trend,2,7,2016-06,14.68%,"3,600,271.36",+14.68%
14,ETS Damped Trend,3,4,2016-03,13.51%,"3,695,955.93",-9.54%



Worst window for each of the nine models:


,model,worst_rank,window_id,train_end,WAPE_pct,RMSE,Bias_pct
0,50/50 Blended,1,3,2016-02,22.76%,"5,768,732.10",-22.76%
3,Calendar + Promotion,1,6,2016-05,33.85%,"7,732,664.69",+33.85%
6,Calendar Only,1,1,2015-12,24.89%,"5,810,011.36",-24.89%
9,ETS Additive,1,9,2016-08,14.41%,"3,474,959.84",+14.41%
12,ETS Damped Trend,1,6,2016-05,15.56%,"3,566,475.04",+15.56%
15,Naive,1,1,2015-12,18.35%,"4,306,266.38",+18.35%
18,Seasonal Index,1,2,2016-01,26.29%,"6,743,810.18",-26.29%
21,Seasonal Naive,1,2,2016-01,37.65%,"9,041,497.93",-37.65%
24,Trend-Adjusted Seasonal Index,1,8,2016-07,43.84%,"10,198,116.38",+43.84%


## Step 8 — Forecast Error Concentration Analysis

Measure how much each historical rolling-origin window
contributes to a model's total absolute forecast error.

Distinguish between:
- Highest window-level WAPE
- Largest total absolute-error contribution

Calculate the concentration of absolute errors in the
three worst-contributing windows for each model.

Forecast windows contain overlapping target months.
Error totals represent evaluation-case errors, not
independent realized business losses.

In [9]:
# Calculate each model's total absolute error
model_total_error_df = (
    window_accuracy_df
    .groupby("model")["MAE"]
    .sum()
    .rename("sum_window_MAE")
    .reset_index()
)

# Each window has exactly three forecast cases,
# so window absolute error = window MAE * 3
error_concentration_df = window_accuracy_df.copy()

error_concentration_df["window_absolute_error"] = (
    error_concentration_df["MAE"]
    * error_concentration_df["forecast_cases"]
)

# Calculate each model's total absolute error
error_concentration_df["model_total_absolute_error"] = (
    error_concentration_df
    .groupby("model")["window_absolute_error"]
    .transform("sum")
)

# Calculate contribution of each window
error_concentration_df["error_contribution_pct"] = (
    error_concentration_df["window_absolute_error"]
    / error_concentration_df["model_total_absolute_error"]
    * 100
)

# Rank windows by absolute error contribution
error_concentration_df["contribution_rank"] = (
    error_concentration_df
    .groupby("model")["window_absolute_error"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Select the three largest contributors per model
top_error_windows_df = (
    error_concentration_df[
        error_concentration_df["contribution_rank"] <= 3
    ]
    .sort_values(["model", "contribution_rank"])
    .copy()
)

# Summarize top-three concentration
top_three_concentration_df = (
    top_error_windows_df
    .groupby("model")
    .agg(
        top_three_error_contribution_pct=(
            "error_contribution_pct", "sum"
        )
    )
    .reset_index()
)

# Preserve overall WAPE model ranking
top_three_concentration_df = (
    top_three_concentration_df
    .set_index("model")
    .loc[model_order]
    .reset_index()
)

# Validate contribution totals
contribution_totals = (
    error_concentration_df
    .groupby("model")["error_contribution_pct"]
    .sum()
)

assert np.allclose(
    contribution_totals.to_numpy(),
    100,
    atol=1e-8
)

assert len(top_error_windows_df) == 27
assert len(top_three_concentration_df) == 9

assert top_three_concentration_df[
    "top_three_error_contribution_pct"
].between(0, 100).all()

# Independently reconcile against the unified dataset
expected_error_totals = (
    accuracy_work_df
    .groupby("model")["absolute_error"]
    .sum()
)

calculated_error_totals = (
    error_concentration_df
    .groupby("model")["window_absolute_error"]
    .sum()
)

assert np.allclose(
    calculated_error_totals.loc[expected_error_totals.index],
    expected_error_totals,
    rtol=1e-10,
    atol=0.01
)

print("Forecast error concentration validation: PASSED")
print("Top-three window records:", len(top_error_windows_df))

print("\nTop-three absolute-error concentration by model:")

display(
    top_three_concentration_df.style.format(
        {"top_three_error_contribution_pct": "{:.2f}%"}
    )
)

print("\nTop-three contributing windows for leading models:")

display(
    top_error_windows_df[
        top_error_windows_df["model"].isin(
            ["50/50 Blended", "ETS Additive", "ETS Damped Trend"]
        )
    ][
        [
            "model",
            "contribution_rank",
            "window_id",
            "train_end",
            "WAPE_pct",
            "window_absolute_error",
            "error_contribution_pct",
        ]
    ].style.format(
        {
            "train_end": "{:%Y-%m}",
            "WAPE_pct": "{:.2f}%",
            "window_absolute_error": "{:,.2f}",
            "error_contribution_pct": "{:.2f}%",
        }
    )
)

Forecast error concentration validation: PASSED
Top-three window records: 27

Top-three absolute-error concentration by model:


,model,top_three_error_contribution_pct
0,50/50 Blended,39.05%
1,ETS Additive,28.05%
2,ETS Damped Trend,28.21%
3,Naive,32.75%
4,Calendar Only,33.64%
5,Calendar + Promotion,38.91%
6,Seasonal Naive,47.12%
7,Seasonal Index,28.38%
8,Trend-Adjusted Seasonal Index,35.82%



Top-three contributing windows for leading models:


,model,contribution_rank,window_id,train_end,WAPE_pct,window_absolute_error,error_contribution_pct
25,50/50 Blended,1,3,2016-02,22.76%,"16,811,103.08",15.94%
13,50/50 Blended,2,2,2016-01,18.20%,"12,927,471.69",12.26%
33,50/50 Blended,3,4,2016-03,15.67%,"11,432,310.31",10.84%
79,ETS Additive,1,9,2016-08,14.41%,"10,246,517.95",9.58%
32,ETS Additive,2,4,2016-03,13.76%,"10,035,091.93",9.38%
22,ETS Additive,3,3,2016-02,13.17%,"9,731,619.15",9.09%
51,ETS Damped Trend,1,6,2016-05,15.56%,"10,601,913.67",9.81%
60,ETS Damped Trend,2,7,2016-06,14.68%,"10,030,251.40",9.28%
31,ETS Damped Trend,3,4,2016-03,13.51%,"9,856,547.19",9.12%


## Step 9 — Rolling-Window Model Win-Rate Analysis

Compare the nine forecasting models within each of the
17 matched rolling-origin windows.

For each window:
- Rank models by window-level WAPE.
- Identify the model with the lowest WAPE.
- Count how many windows each model wins.
- Calculate the percentage of windows won.

Also calculate the average rank of each model across
all windows.

Use deterministic alphabetical tie-breaking for exact
ties and inspect whether any ties occurred.

Win rates describe historical performance only.
Overlapping forecast targets mean that windows are
not statistically independent observations.

In [10]:
# Start with the validated window-level accuracy dataset
win_rate_work_df = window_accuracy_df.copy()

# Rank models by WAPE within each historical window
# Rank 1 = lowest WAPE
win_rate_work_df["window_rank"] = (
    win_rate_work_df
    .groupby("window_id")["WAPE_pct"]
    .rank(method="min", ascending=True)
)

# Check for exact ties in window-level WAPE
tie_counts = (
    win_rate_work_df
    .groupby(["window_id", "WAPE_pct"])
    .size()
)

exact_ties = tie_counts[tie_counts > 1]

# Sort deterministically by window, WAPE, and model name
sorted_window_models_df = (
    win_rate_work_df
    .sort_values(
        ["window_id", "WAPE_pct", "model"]
    )
)

# Select one winning model per window
window_winners_df = (
    sorted_window_models_df
    .groupby("window_id")
    .head(1)
    .copy()
)

# Calculate wins and average ranking
model_win_rates_df = (
    win_rate_work_df
    .groupby("model")
    .agg(
        windows_evaluated=("window_id", "nunique"),
        average_window_rank=("window_rank", "mean"),
    )
    .reset_index()
)

win_counts = (
    window_winners_df["model"]
    .value_counts()
    .rename("windows_won")
)

model_win_rates_df["windows_won"] = (
    model_win_rates_df["model"]
    .map(win_counts)
    .fillna(0)
    .astype(int)
)

model_win_rates_df["win_rate_pct"] = (
    model_win_rates_df["windows_won"]
    / model_win_rates_df["windows_evaluated"]
    * 100
)

# Sort by win count, then average rank
model_win_rates_df = (
    model_win_rates_df
    .sort_values(
        ["windows_won", "average_window_rank"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# Validation
assert len(window_winners_df) == 17

assert window_winners_df["window_id"].nunique() == 17

assert len(model_win_rates_df) == 9

assert model_win_rates_df["windows_won"].sum() == 17

assert model_win_rates_df[
    "windows_evaluated"
].eq(17).all()

assert np.isclose(
    model_win_rates_df["win_rate_pct"].sum(),
    100,
)

print("Model win-rate validation: PASSED")
print("Historical windows:", len(window_winners_df))
print("Total window wins:", model_win_rates_df["windows_won"].sum())
print("Exact WAPE ties:", len(exact_ties))

print("\nModel win-rate comparison:")

display(
    model_win_rates_df.style.format(
        {
            "average_window_rank": "{:.2f}",
            "win_rate_pct": "{:.2f}%",
        }
    )
)

print("\nWinning model by rolling-origin window:")

display(
    window_winners_df[
        [
            "window_id",
            "train_end",
            "model",
            "WAPE_pct",
        ]
    ].sort_values("window_id").style.format(
        {
            "train_end": "{:%Y-%m}",
            "WAPE_pct": "{:.2f}%",
        }
    )
)

Model win-rate validation: PASSED
Historical windows: 17
Total window wins: 17
Exact WAPE ties: 0

Model win-rate comparison:


,model,windows_evaluated,average_window_rank,windows_won,win_rate_pct
0,Naive,17,4.24,5,29.41%
1,ETS Additive,17,3.94,4,23.53%
2,Calendar Only,17,5.18,2,11.76%
3,Seasonal Naive,17,5.65,2,11.76%
4,ETS Damped Trend,17,4.06,1,5.88%
5,50/50 Blended,17,4.12,1,5.88%
6,Calendar + Promotion,17,4.82,1,5.88%
7,Trend-Adjusted Seasonal Index,17,6.12,1,5.88%
8,Seasonal Index,17,6.88,0,0.00%



Winning model by rolling-origin window:


,window_id,train_end,model,WAPE_pct
0,1,2015-12,ETS Additive,6.15%
9,2,2016-01,Naive,6.84%
18,3,2016-02,Naive,10.87%
27,4,2016-03,Naive,7.40%
36,5,2016-04,Calendar Only,5.39%
45,6,2016-05,Seasonal Naive,3.46%
54,7,2016-06,50/50 Blended,2.48%
63,8,2016-07,Seasonal Naive,2.16%
72,9,2016-08,Calendar Only,1.21%
81,10,2016-09,ETS Damped Trend,3.25%


## Step 10 — Paired Comparison of Leading Forecast Models

Compare 50/50 Blended and ETS Additive using the same
17 historical rolling-origin windows.

Calculate the difference in window-level WAPE:

    Blended WAPE - ETS Additive WAPE

Interpretation:
- Negative difference: Blended performs better.
- Positive difference: ETS Additive performs better.
- Zero difference: Both models have equal WAPE.

Count the windows won by each model and summarize
the magnitude of the differences.

These comparisons are descriptive because adjacent
rolling-origin windows have overlapping forecast targets.

In [11]:
# Select the two leading models
leading_pair = [
    "50/50 Blended",
    "ETS Additive",
]

# Prepare matched window-level WAPE values
paired_wape_df = (
    window_accuracy_df[
        window_accuracy_df["model"].isin(leading_pair)
    ]
    .pivot(
        index=["window_id", "train_end"],
        columns="model",
        values="WAPE_pct",
    )
    .reset_index()
)

# Calculate paired WAPE difference
paired_wape_df["WAPE_difference_pp"] = (
    paired_wape_df["50/50 Blended"]
    - paired_wape_df["ETS Additive"]
)

# Identify the winner in each window
paired_wape_df["better_model"] = np.select(
    [
        paired_wape_df["WAPE_difference_pp"] < -1e-10,
        paired_wape_df["WAPE_difference_pp"] > 1e-10,
    ],
    [
        "50/50 Blended",
        "ETS Additive",
    ],
    default="Tie",
)

# Validate paired comparison
assert len(paired_wape_df) == 17

assert paired_wape_df[
    ["50/50 Blended", "ETS Additive"]
].notna().all().all()

assert paired_wape_df["window_id"].nunique() == 17

assert np.isfinite(
    paired_wape_df["WAPE_difference_pp"]
).all()

# Summarize results
paired_win_counts = (
    paired_wape_df["better_model"]
    .value_counts()
)

print("Paired model comparison validation: PASSED")

print("\nPaired window wins:")
print(paired_win_counts.to_string())

print(
    "\nMean WAPE difference (percentage points):",
    round(paired_wape_df["WAPE_difference_pp"].mean(), 4)
)

print(
    "Median WAPE difference (percentage points):",
    round(paired_wape_df["WAPE_difference_pp"].median(), 4)
)

print("\nWindow-by-window comparison:")

display(
    paired_wape_df.style.format(
        {
            "train_end": "{:%Y-%m}",
            "50/50 Blended": "{:.2f}%",
            "ETS Additive": "{:.2f}%",
            "WAPE_difference_pp": "{:+.2f}",
        }
    )
)

Paired model comparison validation: PASSED

Paired window wins:
better_model
ETS Additive     12
50/50 Blended     5

Mean WAPE difference (percentage points): -0.3623
Median WAPE difference (percentage points): 1.1617

Window-by-window comparison:


model,window_id,train_end,50/50 Blended,ETS Additive,WAPE_difference_pp,better_model
0,1,2015-12,8.79%,6.15%,+2.65,ETS Additive
1,2,2016-01,18.20%,13.56%,+4.63,ETS Additive
2,3,2016-02,22.76%,13.17%,+9.58,ETS Additive
3,4,2016-03,15.67%,13.76%,+1.92,ETS Additive
4,5,2016-04,7.29%,10.68%,-3.39,50/50 Blended
5,6,2016-05,3.58%,13.86%,-10.28,50/50 Blended
6,7,2016-06,2.48%,14.19%,-11.71,50/50 Blended
7,8,2016-07,2.98%,12.76%,-9.78,50/50 Blended
8,9,2016-08,4.65%,14.41%,-9.76,50/50 Blended
9,10,2016-09,9.86%,7.03%,+2.83,ETS Additive


## Step 11 — Historical Period Stability Analysis

Compare forecasting performance across earlier and later
rolling-origin windows.

Use a chronological split:
- Earlier origins: windows 1–8
- Later origins: windows 9–17

Evaluate the same nine models in both periods using
pooled WAPE, RMSE, and Bias.

The periods are based on forecast origin rather than
non-overlapping target dates.

This analysis is descriptive and does not constitute
an independent holdout evaluation.

In [12]:
# Work with the existing validated forecast errors
period_work_df = accuracy_work_df.copy()

# Assign each forecast origin to a historical period
period_work_df["historical_period"] = np.where(
    period_work_df["window_id"] <= 8,
    "Earlier origins (1-8)",
    "Later origins (9-17)"
)

# Calculate accuracy separately by period and model
period_accuracy_df = (
    period_work_df
    .groupby(["historical_period", "model"])
    .agg(
        forecast_cases=("forecast_error", "size"),
        windows=("window_id", "nunique"),
        MAE=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        total_absolute_error=("absolute_error", "sum"),
        total_forecast_error=("forecast_error", "sum"),
        total_actual_sales=("actual_sales", "sum"),
    )
    .reset_index()
)

period_accuracy_df["RMSE"] = np.sqrt(
    period_accuracy_df["mean_squared_error"]
)

period_accuracy_df["WAPE_pct"] = (
    period_accuracy_df["total_absolute_error"]
    / period_accuracy_df["total_actual_sales"]
    * 100
)

period_accuracy_df["Bias_pct"] = (
    period_accuracy_df["total_forecast_error"]
    / period_accuracy_df["total_actual_sales"]
    * 100
)

# Validate the expected structure
assert len(period_accuracy_df) == 18

assert period_accuracy_df["model"].nunique() == 9

assert set(period_accuracy_df["windows"]) == {8, 9}

assert set(period_accuracy_df["forecast_cases"]) == {24, 27}

assert period_accuracy_df.groupby(
    "model"
)["forecast_cases"].sum().eq(51).all()

# Create a side-by-side WAPE comparison
period_wape_df = period_accuracy_df.pivot(
    index="model",
    columns="historical_period",
    values="WAPE_pct",
)

period_wape_df = period_wape_df.loc[
    model_order,
    [
        "Earlier origins (1-8)",
        "Later origins (9-17)",
    ]
]

period_wape_df["Change_later_minus_earlier_pp"] = (
    period_wape_df["Later origins (9-17)"]
    - period_wape_df["Earlier origins (1-8)"]
)

print("Historical period stability validation: PASSED")
print("Model-period combinations:", len(period_accuracy_df))
print("Earlier period: 8 windows, 24 cases per model")
print("Later period: 9 windows, 27 cases per model")

print("\nHistorical period WAPE comparison:")

display(
    period_wape_df.style.format(
        {
            "Earlier origins (1-8)": "{:.2f}%",
            "Later origins (9-17)": "{:.2f}%",
            "Change_later_minus_earlier_pp": "{:+.2f}",
        }
    )
)

print("\nLeading models: period-level accuracy and bias")

display(
    period_accuracy_df[
        period_accuracy_df["model"].isin(
            ["50/50 Blended", "ETS Additive", "ETS Damped Trend"]
        )
    ][
        [
            "historical_period",
            "model",
            "forecast_cases",
            "WAPE_pct",
            "RMSE",
            "Bias_pct",
        ]
    ].style.format(
        {
            "WAPE_pct": "{:.2f}%",
            "RMSE": "{:,.2f}",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

Historical period stability validation: PASSED
Model-period combinations: 18
Earlier period: 8 windows, 24 cases per model
Later period: 9 windows, 27 cases per model

Historical period WAPE comparison:


historical_period,Earlier origins (1-8),Later origins (9-17),Change_later_minus_earlier_pp
model,,,
50/50 Blended,10.41%,6.71%,-3.70
ETS Additive,12.28%,5.42%,-6.86
ETS Damped Trend,12.53%,5.38%,-7.15
Naive,8.70%,8.89%,+0.19
Calendar Only,11.71%,8.64%,-3.07
Calendar + Promotion,19.80%,6.15%,-13.65
Seasonal Naive,19.75%,7.53%,-12.22
Seasonal Index,15.75%,13.53%,-2.22
Trend-Adjusted Seasonal Index,25.46%,14.42%,-11.04



Leading models: period-level accuracy and bias


,historical_period,model,forecast_cases,WAPE_pct,RMSE,Bias_pct
0,Earlier origins (1-8),50/50 Blended,24,10.41%,"3,296,481.17",-7.94%
3,Earlier origins (1-8),ETS Additive,24,12.28%,"3,237,059.37",+1.49%
4,Earlier origins (1-8),ETS Damped Trend,24,12.53%,"3,288,074.90",+2.04%
9,Later origins (9-17),50/50 Blended,27,6.71%,"2,119,429.32",-5.25%
12,Later origins (9-17),ETS Additive,27,5.42%,"1,700,448.18",+0.53%
13,Later origins (9-17),ETS Damped Trend,27,5.38%,"1,632,452.06",-1.09%


## Step 12 — Evidence-Based Forecast Model Selection Scorecard

Consolidate validated model evaluation results into a
single model-selection scorecard.

Include:
- Overall WAPE and RMSE
- Overall forecast bias
- Mean and standard deviation of window-level WAPE
- Historical window win rate
- Worst-window WAPE
- Top-three absolute-error concentration
- Earlier and later period WAPE

Use this scorecard to identify accuracy, stability,
bias, and historical-period trade-offs.

Do not assign arbitrary weighted scores or claim
production readiness based on historical backtests.

In [13]:
# Start with the validated overall model accuracy results
model_selection_df = nine_model_accuracy_df[
    ["model", "MAE", "RMSE", "WAPE_pct", "Bias_pct"]
].copy()

# Add rolling-window robustness
model_selection_df = model_selection_df.merge(
    window_robustness_df[
        [
            "model",
            "mean_window_WAPE",
            "median_window_WAPE",
            "std_window_WAPE",
            "max_window_WAPE",
        ]
    ],
    on="model",
    validate="one_to_one",
)

# Add model win rates
model_selection_df = model_selection_df.merge(
    model_win_rates_df[
        [
            "model",
            "windows_won",
            "win_rate_pct",
            "average_window_rank",
        ]
    ],
    on="model",
    validate="one_to_one",
)

# Add absolute-error concentration
model_selection_df = model_selection_df.merge(
    top_three_concentration_df,
    on="model",
    validate="one_to_one",
)

# Add historical period WAPE
period_columns_df = period_wape_df[
    [
        "Earlier origins (1-8)",
        "Later origins (9-17)",
    ]
].rename(
    columns={
        "Earlier origins (1-8)": "earlier_WAPE_pct",
        "Later origins (9-17)": "later_WAPE_pct",
    }
).reset_index()

model_selection_df = model_selection_df.merge(
    period_columns_df,
    on="model",
    validate="one_to_one",
)

# Preserve overall WAPE ranking
model_selection_df = (
    model_selection_df
    .set_index("model")
    .loc[model_order]
    .reset_index()
)

# Validate the scorecard
assert len(model_selection_df) == 9
assert model_selection_df["model"].nunique() == 9
assert not model_selection_df.isna().any().any()

assert np.allclose(
    model_selection_df["WAPE_pct"],
    nine_model_accuracy_df["WAPE_pct"],
)

assert model_selection_df["windows_won"].sum() == 17

assert np.allclose(
    model_selection_df["top_three_error_contribution_pct"],
    top_three_concentration_df[
        "top_three_error_contribution_pct"
    ],
)

print("Model selection scorecard validation: PASSED")
print("Models evaluated:", len(model_selection_df))
print("Scorecard columns:", len(model_selection_df.columns))

print("\nLeading candidate comparison:")

leading_candidate_df = model_selection_df[
    model_selection_df["model"].isin(
        [
            "50/50 Blended",
            "ETS Additive",
            "ETS Damped Trend",
            "Naive",
        ]
    )
].copy()

display(
    leading_candidate_df[
        [
            "model",
            "WAPE_pct",
            "RMSE",
            "Bias_pct",
            "std_window_WAPE",
            "windows_won",
            "max_window_WAPE",
            "top_three_error_contribution_pct",
            "earlier_WAPE_pct",
            "later_WAPE_pct",
        ]
    ].style.format(
        {
            "WAPE_pct": "{:.2f}%",
            "RMSE": "{:,.2f}",
            "Bias_pct": "{:+.2f}%",
            "std_window_WAPE": "{:.2f}",
            "max_window_WAPE": "{:.2f}%",
            "top_three_error_contribution_pct": "{:.2f}%",
            "earlier_WAPE_pct": "{:.2f}%",
            "later_WAPE_pct": "{:.2f}%",
        }
    )
)

Model selection scorecard validation: PASSED
Models evaluated: 9
Scorecard columns: 15

Leading candidate comparison:


,model,WAPE_pct,RMSE,Bias_pct,std_window_WAPE,windows_won,max_window_WAPE,top_three_error_contribution_pct,earlier_WAPE_pct,later_WAPE_pct
0,50/50 Blended,8.36%,"2,737,131.72",-6.45%,5.72,1,22.76%,39.05%,10.41%,6.71%
1,ETS Additive,8.48%,"2,542,024.93",+0.96%,4.66,4,14.41%,28.05%,12.28%,5.42%
2,ETS Damped Trend,8.56%,"2,549,228.14",+0.30%,4.64,1,15.56%,28.21%,12.53%,5.38%
3,Naive,8.80%,"2,735,640.76",-0.59%,4.83,5,18.35%,32.75%,8.70%,8.89%


## Step 13 — Model Selection Decision and Business Interpretation

### Recommendation

Recommend **ETS Additive** as the primary candidate for further
validation of aggregate monthly retail sales forecasting.

Retain **50/50 Blended** and **ETS Damped Trend** as challenger models.

### Evidence

- 50/50 Blended achieved the lowest pooled WAPE (8.36%),
  compared with 8.48% for ETS Additive.
- ETS Additive achieved the lowest RMSE (2,542,024.93).
- ETS Additive had +0.96% aggregate bias, compared with
  -6.45% for Blended.
- ETS Additive had lower window-level WAPE variability
  (4.66 percentage points versus 5.72).
- ETS Additive's worst-window WAPE was 14.41%, compared
  with 22.76% for Blended.
- ETS Additive outperformed Blended in 12 of 17 matched
  rolling-origin windows.
- ETS Damped Trend had the smallest absolute aggregate
  bias (+0.30%).

### Retail Business Implications

ETS Additive offers a balanced historical accuracy and
stability profile for aggregate monthly sales planning.

Blended remains relevant when minimizing pooled absolute
error is the primary objective.

ETS Damped Trend remains relevant when minimizing aggregate
bias is especially important.

### Limitations

- Results use 17 rolling-origin windows and 51 overlapping
  forecast cases per model.
- Overlapping target months mean evaluation cases are
  not independent.
- Historical superiority does not guarantee future performance.
- The target is Sales, not verified Net Sales.
- Aggregate forecasts do not directly establish SKU-level
  replenishment requirements or stockout risks.
- No independent untouched final test period was evaluated.
- Promotion-related results do not establish causation.

### Next Validation Requirements

Before production consideration:

1. Establish business acceptance thresholds.
2. Evaluate genuinely unseen future periods.
3. Assess bias and errors by forecast horizon.
4. Document retraining and monitoring procedures.
5. Obtain stakeholder approval.

### Decision Status

- **ETS Additive:** Primary candidate for further validation.
- **50/50 Blended:** Challenger model.
- **ETS Damped Trend:** Challenger model.
- **Production deployment:** Not approved.

## Step 14 — Export Validated Model Accuracy Report

Export the validated nine-model accuracy comparison to CSV.

Before exporting:
- Confirm that all nine models are present.
- Confirm that every model has 51 matched forecast cases.
- Confirm that accuracy metrics contain no missing values.

After exporting:
- Read the CSV back into Python.
- Reconcile the exported results with the notebook results.
- Confirm that the file can be reused in downstream reporting.

The exported report contains historical backtest metrics,
not production forecast performance.

In [14]:
# Create a dedicated Phase 6.4 reporting folder
robustness_report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_robustness"
)

robustness_report_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Define the output file
accuracy_export_path = (
    robustness_report_dir
    / "nine_model_accuracy_comparison.csv"
)

# Validate before exporting
assert len(nine_model_accuracy_df) == 9

assert nine_model_accuracy_df[
    "forecast_cases"
].eq(51).all()

assert not nine_model_accuracy_df.isna().any().any()

# Export the report
nine_model_accuracy_df.to_csv(
    accuracy_export_path,
    index=False,
    float_format="%.10f"
)

# Read the exported report back
accuracy_reloaded_df = pd.read_csv(
    accuracy_export_path
)

# Validate the exported structure
assert accuracy_reloaded_df.shape == (
    nine_model_accuracy_df.shape
)

assert (
    accuracy_reloaded_df["model"].tolist()
    == nine_model_accuracy_df["model"].tolist()
)

assert accuracy_reloaded_df[
    "forecast_cases"
].eq(51).all()

# Reconcile all numeric results
metric_columns = [
    "MAE",
    "RMSE",
    "WAPE_pct",
    "Bias_pct",
]

assert np.allclose(
    accuracy_reloaded_df[metric_columns].to_numpy(),
    nine_model_accuracy_df[metric_columns].to_numpy(),
    rtol=0,
    atol=1e-8
)

print("Nine-model accuracy export validation: PASSED")
print("Exported file:", accuracy_export_path)
print("Exported rows:", len(accuracy_reloaded_df))
print("Exported columns:", len(accuracy_reloaded_df.columns))

print("\nReloaded accuracy report:")
display(accuracy_reloaded_df)

Nine-model accuracy export validation: PASSED
Exported file: d:\GitHub\retail-sales-forecasting\reports\forecasting_robustness\nine_model_accuracy_comparison.csv
Exported rows: 9
Exported columns: 6

Reloaded accuracy report:


,model,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,50/50 Blended,51,2.067405e+06,2.737132e+06,8.356751,-6.451589
1,ETS Additive,51,2.098242e+06,2.542025e+06,8.481401,0.961165
2,ETS Damped Trend,51,2.118860e+06,2.549228e+06,8.564740,0.304786
3,Naive,51,2.178014e+06,2.735641e+06,8.803852,-0.592666
4,Calendar Only,51,2.475681e+06,3.007249e+06,10.007066,-7.852292
5,Calendar + Promotion,51,3.028155e+06,3.789357e+06,12.240245,-1.343830
6,Seasonal Naive,51,3.210528e+06,4.529398e+06,12.977425,-12.310513
7,Seasonal Index,51,3.591289e+06,4.136903e+06,14.516514,-8.090842
8,Trend-Adjusted Seasonal Index,51,4.786033e+06,6.151133e+06,19.345845,12.575725


## Step 15 — Export Forecast Horizon Accuracy Report

Export the validated nine-model forecast horizon comparison.

The report contains:
- Nine forecasting models
- Three forecast horizons
- Seventeen matched forecast cases per model-horizon
- MAE, RMSE, WAPE, and Bias

Validate the exported file by reloading it and reconciling
all numerical metrics against the notebook calculations.

The results describe historical rolling-origin backtests,
not guaranteed future forecasting performance.

In [15]:
# Define the forecast horizon report path
horizon_export_path = (
    robustness_report_dir
    / "forecast_horizon_accuracy.csv"
)

# Validate the source DataFrame
assert len(horizon_accuracy_df) == 27

assert horizon_accuracy_df[
    "forecast_cases"
].eq(17).all()

assert horizon_accuracy_df[
    "model"
].nunique() == 9

assert set(
    horizon_accuracy_df["forecast_horizon"]
) == {1, 2, 3}

assert not horizon_accuracy_df.isna().any().any()

# Export the validated horizon report
horizon_accuracy_df.to_csv(
    horizon_export_path,
    index=False,
    float_format="%.10f"
)

# Reload the saved CSV
horizon_reloaded_df = pd.read_csv(
    horizon_export_path
)

# Validate structure
assert horizon_reloaded_df.shape == (
    horizon_accuracy_df.shape
)

# Verify matching model and horizon keys
assert horizon_reloaded_df[
    ["model", "forecast_horizon"]
].equals(
    horizon_accuracy_df[
        ["model", "forecast_horizon"]
    ].reset_index(drop=True)
)

# Reconcile numerical metrics
horizon_metric_columns = [
    "MAE",
    "RMSE",
    "WAPE_pct",
    "Bias_pct",
]

assert np.allclose(
    horizon_reloaded_df[
        horizon_metric_columns
    ].to_numpy(),
    horizon_accuracy_df[
        horizon_metric_columns
    ].to_numpy(),
    rtol=0,
    atol=1e-8
)

# Verify 17 forecast cases per combination
assert horizon_reloaded_df[
    "forecast_cases"
].eq(17).all()

print("Forecast horizon export validation: PASSED")

print("Exported file:", horizon_export_path)
print("Exported rows:", len(horizon_reloaded_df))
print("Exported columns:", len(horizon_reloaded_df.columns))

print("\nForecast horizon report preview:")

display(
    horizon_reloaded_df.head(9).style.format(
        {
            "MAE": "{:,.2f}",
            "RMSE": "{:,.2f}",
            "WAPE_pct": "{:.2f}%",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

Forecast horizon export validation: PASSED
Exported file: d:\GitHub\retail-sales-forecasting\reports\forecasting_robustness\forecast_horizon_accuracy.csv
Exported rows: 27
Exported columns: 7

Forecast horizon report preview:


,model,forecast_horizon,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,ETS Damped Trend,1,17,"1,907,408.08","2,341,342.18",7.76%,-0.30%
1,ETS Additive,1,17,"1,954,976.05","2,406,598.99",7.96%,+0.39%
2,Naive,1,17,"2,015,108.34","2,386,521.98",8.20%,+0.08%
3,50/50 Blended,1,17,"2,137,409.25","2,752,046.60",8.70%,-6.79%
4,Calendar Only,1,17,"2,568,679.21","3,223,319.42",10.45%,-9.27%
5,Calendar + Promotion,1,17,"2,875,126.75","3,586,726.16",11.70%,-2.71%
6,Seasonal Naive,1,17,"3,521,719.88","4,934,490.91",14.33%,-13.66%
7,Seasonal Index,1,17,"3,612,538.39","4,119,864.29",14.70%,-7.99%
8,Trend-Adjusted Seasonal Index,1,17,"4,839,347.10","5,936,459.50",19.69%,+12.57%


## Step 16 — Export Rolling-Window Robustness Report

Export the validated rolling-window accuracy results.

Each record represents one forecasting model evaluated
across three forecast horizons from one historical origin.

Expected coverage:
- 9 forecasting models
- 17 rolling-origin windows
- 153 model-window combinations
- 3 forecast cases per combination

Reload the CSV and reconcile its values against the
original notebook calculations.

Overlapping target months mean these windows are not
statistically independent observations.

In [16]:
# Define the output file
window_export_path = (
    robustness_report_dir
    / "rolling_window_accuracy.csv"
)

# Validate the original DataFrame
assert len(window_accuracy_df) == 153

assert window_accuracy_df["model"].nunique() == 9

assert window_accuracy_df["window_id"].nunique() == 17

assert window_accuracy_df["forecast_cases"].eq(3).all()

assert not window_accuracy_df.isna().any().any()

# Export the validated report
window_accuracy_df.to_csv(
    window_export_path,
    index=False,
    float_format="%.10f",
    date_format="%Y-%m-%d"
)

# Reload the exported report
window_reloaded_df = pd.read_csv(
    window_export_path,
    parse_dates=["train_end"]
)

# Validate structure
assert window_reloaded_df.shape == window_accuracy_df.shape

# Validate row identity
key_columns = ["window_id", "train_end", "model"]

assert window_reloaded_df[
    key_columns
].reset_index(drop=True).equals(
    window_accuracy_df[
        key_columns
    ].reset_index(drop=True)
)

# Reconcile numeric accuracy metrics
window_metric_columns = [
    "MAE",
    "RMSE",
    "WAPE_pct",
    "Bias_pct",
]

assert np.allclose(
    window_reloaded_df[
        window_metric_columns
    ].to_numpy(),
    window_accuracy_df[
        window_metric_columns
    ].to_numpy(),
    rtol=0,
    atol=1e-8
)

# Validate every window contains all nine models
models_per_window = (
    window_reloaded_df
    .groupby("window_id")["model"]
    .nunique()
)

assert models_per_window.eq(9).all()

# Validate each model has 17 windows
windows_per_model = (
    window_reloaded_df
    .groupby("model")["window_id"]
    .nunique()
)

assert windows_per_model.eq(17).all()

print("Rolling-window report export validation: PASSED")
print("Exported file:", window_export_path)
print("Exported rows:", len(window_reloaded_df))
print("Exported columns:", len(window_reloaded_df.columns))

print("\nRolling-window report preview:")

display(
    window_reloaded_df.head(9).style.format(
        {
            "train_end": "{:%Y-%m}",
            "MAE": "{:,.2f}",
            "RMSE": "{:,.2f}",
            "WAPE_pct": "{:.2f}%",
            "Bias_pct": "{:+.2f}%",
        }
    )
)

Rolling-window report export validation: PASSED
Exported file: d:\GitHub\retail-sales-forecasting\reports\forecasting_robustness\rolling_window_accuracy.csv
Exported rows: 153
Exported columns: 8

Rolling-window report preview:


,window_id,train_end,model,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,1,2015-12,ETS Additive,3,"1,414,984.65","1,677,958.29",6.15%,-3.00%
1,1,2015-12,ETS Damped Trend,3,"1,456,322.45","1,821,712.32",6.33%,-5.83%
2,1,2015-12,50/50 Blended,3,"2,024,019.78","2,120,787.66",8.79%,-8.79%
3,1,2015-12,Trend-Adjusted Seasonal Index,3,"2,941,563.64","3,441,307.47",12.78%,-12.78%
4,1,2015-12,Naive,3,"4,224,983.48","4,306,266.38",18.35%,+18.35%
5,1,2015-12,Seasonal Index,3,"5,559,251.79","5,758,106.55",24.15%,-24.15%
6,1,2015-12,Calendar Only,3,"5,730,170.30","5,810,011.36",24.89%,-24.89%
7,1,2015-12,Calendar + Promotion,3,"6,892,555.87","6,949,473.14",29.94%,-29.94%
8,1,2015-12,Seasonal Naive,3,"8,273,023.05","8,297,256.00",35.94%,-35.94%


## Step 17 — Export the Model Selection Scorecard

Export the validated nine-model selection scorecard.

The scorecard consolidates:
- Overall MAE, RMSE, WAPE, and Bias
- Rolling-window accuracy and variability
- Historical window win counts and average ranks
- Top-three window absolute-error concentration
- Earlier and later historical period WAPE

Reload the CSV and reconcile all values with the
original scorecard.

This report supports an evidence-based model selection
decision; it does not constitute production approval.

In [17]:
# Define the scorecard output path
scorecard_export_path = (
    robustness_report_dir
    / "model_selection_scorecard.csv"
)

# Validate the source scorecard
assert len(model_selection_df) == 9
assert len(model_selection_df.columns) == 15

assert model_selection_df["model"].nunique() == 9

assert not model_selection_df.isna().any().any()

# Export the scorecard
model_selection_df.to_csv(
    scorecard_export_path,
    index=False,
    float_format="%.10f"
)

# Reload the saved CSV
scorecard_reloaded_df = pd.read_csv(
    scorecard_export_path
)

# Validate structure
assert scorecard_reloaded_df.shape == (
    model_selection_df.shape
)

# Validate column names and model order
assert (
    scorecard_reloaded_df.columns.tolist()
    == model_selection_df.columns.tolist()
)

assert (
    scorecard_reloaded_df["model"].tolist()
    == model_selection_df["model"].tolist()
)

# Reconcile all numeric values
numeric_columns = model_selection_df.select_dtypes(
    include="number"
).columns.tolist()

assert np.allclose(
    scorecard_reloaded_df[numeric_columns].to_numpy(),
    model_selection_df[numeric_columns].to_numpy(),
    rtol=0,
    atol=1e-8
)

# Validate expected window win totals
assert scorecard_reloaded_df[
    "windows_won"
].sum() == 17

# Validate model ranking by overall WAPE
assert scorecard_reloaded_df[
    "WAPE_pct"
].is_monotonic_increasing

print("Model selection scorecard export validation: PASSED")

print("Exported file:", scorecard_export_path)
print("Exported rows:", len(scorecard_reloaded_df))
print("Exported columns:", len(scorecard_reloaded_df.columns))

print("\nExported scorecard preview:")

display(
    scorecard_reloaded_df[
        [
            "model",
            "WAPE_pct",
            "RMSE",
            "Bias_pct",
            "windows_won",
            "std_window_WAPE",
        ]
    ].style.format(
        {
            "WAPE_pct": "{:.2f}%",
            "RMSE": "{:,.2f}",
            "Bias_pct": "{:+.2f}%",
            "std_window_WAPE": "{:.2f}",
        }
    )
)

Model selection scorecard export validation: PASSED
Exported file: d:\GitHub\retail-sales-forecasting\reports\forecasting_robustness\model_selection_scorecard.csv
Exported rows: 9
Exported columns: 15

Exported scorecard preview:


,model,WAPE_pct,RMSE,Bias_pct,windows_won,std_window_WAPE
0,50/50 Blended,8.36%,"2,737,131.72",-6.45%,1,5.72
1,ETS Additive,8.48%,"2,542,024.93",+0.96%,4,4.66
2,ETS Damped Trend,8.56%,"2,549,228.14",+0.30%,1,4.64
3,Naive,8.80%,"2,735,640.76",-0.59%,5,4.83
4,Calendar Only,10.01%,"3,007,249.48",-7.85%,2,5.99
5,Calendar + Promotion,12.24%,"3,789,357.27",-1.34%,1,9.58
6,Seasonal Naive,12.98%,"4,529,398.49",-12.31%,2,12.16
7,Seasonal Index,14.52%,"4,136,902.96",-8.09%,0,6.10
8,Trend-Adjusted Seasonal Index,19.35%,"6,151,133.27",+12.58%,1,15.06


## Step 18 — Final Cross-Report Reconciliation

Validate consistency across all four exported Phase 6.4
reports.

Reconstruct pooled WAPE, RMSE, and Bias from the underlying
forecast errors and compare against the exported accuracy
report.

Verify:
- Nine models and 51 matched cases per model.
- Three horizons and 17 cases per model-horizon.
- Seventeen windows and three cases per model-window.
- Consistent model selection scorecard metrics.
- No duplicate keys or missing numeric metrics.

Only validated reports will be included in the GitHub
milestone.

In [18]:
# Reload all four saved reports
accuracy_check_df = pd.read_csv(accuracy_export_path)
horizon_check_df = pd.read_csv(horizon_export_path)
window_check_df = pd.read_csv(
    window_export_path,
    parse_dates=["train_end"]
)
scorecard_check_df = pd.read_csv(scorecard_export_path)

# 1. Validate unique keys
assert not accuracy_check_df.duplicated(["model"]).any()

assert not horizon_check_df.duplicated(
    ["model", "forecast_horizon"]
).any()

assert not window_check_df.duplicated(
    ["model", "window_id"]
).any()

assert not scorecard_check_df.duplicated(["model"]).any()

# 2. Verify model coverage
expected_models = set(unified_forecasts_df["model"])

for report in [
    accuracy_check_df,
    horizon_check_df,
    window_check_df,
    scorecard_check_df,
]:
    assert set(report["model"]) == expected_models

# 3. Verify case counts
assert accuracy_check_df["forecast_cases"].eq(51).all()

assert horizon_check_df["forecast_cases"].eq(17).all()

assert window_check_df["forecast_cases"].eq(3).all()

assert horizon_check_df.groupby("model")[
    "forecast_cases"
].sum().eq(51).all()

assert window_check_df.groupby("model")[
    "forecast_cases"
].sum().eq(51).all()

# 4. Reconstruct metrics from individual forecasts
reconstructed_df = (
    accuracy_work_df
    .groupby("model")
    .agg(
        cases=("forecast_error", "size"),
        total_absolute_error=("absolute_error", "sum"),
        total_squared_error=("squared_error", "sum"),
        total_forecast_error=("forecast_error", "sum"),
        total_actual_sales=("actual_sales", "sum"),
    )
)

reconstructed_df["MAE"] = (
    reconstructed_df["total_absolute_error"]
    / reconstructed_df["cases"]
)

reconstructed_df["RMSE"] = np.sqrt(
    reconstructed_df["total_squared_error"]
    / reconstructed_df["cases"]
)

reconstructed_df["WAPE_pct"] = (
    reconstructed_df["total_absolute_error"]
    / reconstructed_df["total_actual_sales"]
    * 100
)

reconstructed_df["Bias_pct"] = (
    reconstructed_df["total_forecast_error"]
    / reconstructed_df["total_actual_sales"]
    * 100
)

# 5. Compare reconstructed metrics with exported accuracy
metric_cols = ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]

accuracy_aligned_df = accuracy_check_df.set_index("model")
scorecard_aligned_df = scorecard_check_df.set_index("model")

for metric in metric_cols:
    assert np.allclose(
        reconstructed_df.loc[
            accuracy_aligned_df.index, metric
        ].to_numpy(),
        accuracy_aligned_df[metric].to_numpy(),
        rtol=0,
        atol=1e-8,
    )

    assert np.allclose(
        accuracy_aligned_df.loc[
            scorecard_aligned_df.index, metric
        ].to_numpy(),
        scorecard_aligned_df[metric].to_numpy(),
        rtol=0,
        atol=1e-8,
    )

# 6. Reconstruct overall WAPE from horizon-level results
# Each horizon has its own actual-sales denominator.
horizon_denominators = (
    accuracy_work_df
    .groupby(["model", "forecast_horizon"])["actual_sales"]
    .sum()
)

horizon_aligned_df = horizon_check_df.set_index(
    ["model", "forecast_horizon"]
)

horizon_absolute_error = (
    horizon_aligned_df["WAPE_pct"]
    / 100
    * horizon_denominators.loc[horizon_aligned_df.index]
)

reconstructed_horizon_wape = (
    horizon_absolute_error.groupby(level="model").sum()
    / horizon_denominators.groupby(level="model").sum()
    * 100
)

assert np.allclose(
    reconstructed_horizon_wape.loc[
        accuracy_aligned_df.index
    ].to_numpy(),
    accuracy_aligned_df["WAPE_pct"].to_numpy(),
    rtol=0,
    atol=1e-8,
)

# 7. Verify all exported numeric metrics are finite
for report in [
    accuracy_check_df,
    horizon_check_df,
    window_check_df,
    scorecard_check_df,
]:
    numeric_values = report.select_dtypes(
        include="number"
    ).to_numpy()

    assert np.isfinite(numeric_values).all()

print("Final cross-report reconciliation: PASSED")
print("Validated reports: 4")
print("Forecasting models: 9")
print("Matched forecast cases per model: 51")
print("Horizon combinations: 27")
print("Model-window combinations: 153")
print("Scorecard records: 9")

print("\nOverall WAPE reconciliation:")

display(
    pd.DataFrame({
        "model": accuracy_aligned_df.index,
        "exported_WAPE_pct": (
            accuracy_aligned_df["WAPE_pct"].to_numpy()
        ),
        "reconstructed_WAPE_pct": (
            reconstructed_df.loc[
                accuracy_aligned_df.index,
                "WAPE_pct"
            ].to_numpy()
        ),
    }).style.format({
        "exported_WAPE_pct": "{:.4f}%",
        "reconstructed_WAPE_pct": "{:.4f}%",
    })
)

Final cross-report reconciliation: PASSED
Validated reports: 4
Forecasting models: 9
Matched forecast cases per model: 51
Horizon combinations: 27
Model-window combinations: 153
Scorecard records: 9

Overall WAPE reconciliation:


,model,exported_WAPE_pct,reconstructed_WAPE_pct
0,50/50 Blended,8.3568%,8.3568%
1,ETS Additive,8.4814%,8.4814%
2,ETS Damped Trend,8.5647%,8.5647%
3,Naive,8.8039%,8.8039%
4,Calendar Only,10.0071%,10.0071%
5,Calendar + Promotion,12.2402%,12.2402%
6,Seasonal Naive,12.9774%,12.9774%
7,Seasonal Index,14.5165%,14.5165%
8,Trend-Adjusted Seasonal Index,19.3458%,19.3458%
